# 03 — Model Training: LegalBERT vs DistilBERT on CUAD

This notebook fine-tunes **two models** on the CUAD commercial-lease clause-extraction task:

| Model | HuggingFace ID | Role |
|---|---|---|
| **LegalBERT** | `nlpaueb/legal-bert-base-uncased` | Primary — pre-trained on legal text |
| **DistilBERT** | `distilbert-base-uncased` | Baseline — smaller, general-purpose BERT |

Both are trained as **extractive QA** models: given a clause-type question and a
contract passage, predict the start/end token positions of the answer span.

### Why chunking is required
Legal contracts have a median of ~7,800 tokens (post-cleaning), well beyond
LegalBERT's 512-token limit.  We use a **sliding-window** approach:
- Window size: `MAX_LENGTH = 384` tokens
- Overlap (stride): `STRIDE = 128` tokens
- For each training row we keep **only the most relevant window** (the one
  containing the answer span for answerable rows, or the first window for
  unanswerable rows), giving roughly 1 training example per data row.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForQuestionAnswering,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

# Make src/ importable from within the notebooks/ directory
sys.path.insert(0, str(Path("../src").resolve()))

print(f"PyTorch version : {torch.__version__}")
print(f"MPS available   : {torch.backends.mps.is_available()}")
print(f"CUDA available  : {torch.cuda.is_available()}")


/Users/mustafayunus/Desktop/leaseiq-ml/leaseiq-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version : 2.12.1
MPS available   : True
CUDA available  : False


In [2]:
# ── File paths ────────────────────────────────────────────────────────────
DATA_PATH      = Path("../data/processed/cuad_final.csv")
MODELS_DIR     = Path("../models")
LEGALBERT_OUT  = MODELS_DIR / "legalbert-cuad"
DISTILBERT_OUT = MODELS_DIR / "distilbert-cuad"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ── Model identifiers ─────────────────────────────────────────────────────
LEGALBERT_NAME  = "nlpaueb/legal-bert-base-uncased"
DISTILBERT_NAME = "distilbert-base-uncased"

# ── Tokenisation hyper-parameters ─────────────────────────────────────────
MAX_LENGTH = 384   # token window size; 384 keeps memory manageable on M-series
STRIDE     = 128   # overlap between consecutive windows

# ── Training hyper-parameters ─────────────────────────────────────────────
NUM_TRAIN_CONTRACTS = 50   # contracts from the train split
NUM_EPOCHS          = 2
BATCH_SIZE          = 8
LEARNING_RATE       = 2e-5
WEIGHT_DECAY        = 0.01

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Config:")
print(f"  Training contracts : {NUM_TRAIN_CONTRACTS}")
print(f"  Epochs             : {NUM_EPOCHS}")
print(f"  Batch size         : {BATCH_SIZE}")
print(f"  Max length         : {MAX_LENGTH} tokens")
print(f"  Stride             : {STRIDE} tokens")
print(f"  Learning rate      : {LEARNING_RATE}")


Config:
  Training contracts : 50
  Epochs             : 2
  Batch size         : 8
  Max length         : 384 tokens
  Stride             : 128 tokens
  Learning rate      : 2e-05


## Load and Sample Training Data

In [3]:
df       = pd.read_csv(DATA_PATH)
train_df = df[df["split"] == "train"].reset_index(drop=True)

all_contracts = train_df["contract_id"].unique()
rng           = np.random.default_rng(RANDOM_SEED)
sampled_ids   = rng.choice(all_contracts, size=NUM_TRAIN_CONTRACTS, replace=False)
train_sample  = train_df[train_df["contract_id"].isin(sampled_ids)].reset_index(drop=True)

print(f"Total train contracts available : {len(all_contracts)}")
print(f"Contracts sampled               : {len(sampled_ids)}")
print(f"QA pairs in sample              : {len(train_sample):,}")
print(f"  Answerable                    : {train_sample['is_answerable'].sum():,}")
print(f"  Unanswerable                  : {(~train_sample['is_answerable']).sum():,}")
print(f"Categories covered              : {train_sample[train_sample['is_answerable']]['category'].nunique()} / 41")


Total train contracts available : 408
Contracts sampled               : 50
QA pairs in sample              : 2,729
  Answerable                    : 1,348
  Unanswerable                  : 1,381
Categories covered              : 41 / 41


## Sliding-Window Tokenisation

The `make_qa_examples` function below encodes each (clause-type, contract) pair
into overlapping 384-token windows.  For each training row it returns at most
**one** tokenised example — the chunk most relevant to the answer — instead of
the full ~20 chunks that naive tokenisation would produce.

This is the standard approach used in the SQuAD fine-tuning recipes
(HuggingFace QA tutorial), adapted here for speed on laptop hardware.


In [4]:
def make_qa_examples(df, tokenizer, max_length=384, stride=128,
                     include_token_type_ids=True):
    """
    Convert DataFrame rows into tokenised QA examples (one per row).

    Answerable rows  : keep the first window fully containing the answer span;
                       record exact token start/end positions.
    Unanswerable rows: keep the first window with start=end=0 (CLS convention).

    Args:
        df                    : rows with category, context, answer_text,
                                answer_start, is_answerable columns.
        tokenizer             : HuggingFace tokenizer.
        max_length            : token window size.
        stride                : window overlap in tokens.
        include_token_type_ids: set False for DistilBERT (no token_type_ids).

    Returns:
        List of dicts ready for QADataset.
    """
    examples = []
    skipped  = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc="tokenising"):
        encoding = tokenizer(
            row["category"],   # question = clause type label
            row["context"],    # context  = contract text
            max_length=max_length,
            truncation="only_second",         # protect question from truncation
            stride=stride,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
            padding="max_length",
        )

        placed = False
        for chunk_idx in range(len(encoding["input_ids"])):
            seq_ids = encoding.sequence_ids(chunk_idx)
            offsets = encoding["offset_mapping"][chunk_idx]

            # Context token indices within this chunk (sequence_id == 1)
            ctx_tokens = [i for i, s in enumerate(seq_ids) if s == 1]
            if not ctx_tokens:
                continue
            ctx_start, ctx_end = ctx_tokens[0], ctx_tokens[-1]

            # ── Unanswerable: first chunk only, mark as CLS ───────────────
            if not row["is_answerable"]:
                if chunk_idx > 0:
                    continue
                ex = {
                    "input_ids":       encoding["input_ids"][chunk_idx],
                    "attention_mask":  encoding["attention_mask"][chunk_idx],
                    "start_positions": 0,
                    "end_positions":   0,
                }
                if include_token_type_ids and "token_type_ids" in encoding.keys():
                    ex["token_type_ids"] = encoding["token_type_ids"][chunk_idx]
                examples.append(ex)
                placed = True
                break

            # ── Answerable: find the chunk that contains the answer span ──
            a_start = int(row["answer_start"])
            a_end   = a_start + len(str(row["answer_text"])) - 1

            chunk_char_start = offsets[ctx_start][0]
            chunk_char_end   = offsets[ctx_end][1]

            if chunk_char_start > a_start or chunk_char_end < a_end:
                continue  # answer is not in this window; try the next

            # Walk forward to find the start token
            tok_s = ctx_start
            while tok_s <= ctx_end and offsets[tok_s][0] <= a_start:
                tok_s += 1
            tok_s -= 1

            # Walk backward to find the end token
            tok_e = ctx_end
            while tok_e >= ctx_start and offsets[tok_e][1] >= a_end + 1:
                tok_e -= 1
            tok_e += 1

            if tok_s < 0 or tok_e >= max_length or tok_s > tok_e:
                skipped += 1
                continue  # degenerate span; skip

            ex = {
                "input_ids":       encoding["input_ids"][chunk_idx],
                "attention_mask":  encoding["attention_mask"][chunk_idx],
                "start_positions": tok_s,
                "end_positions":   tok_e,
            }
            if include_token_type_ids and "token_type_ids" in encoding.keys():
                ex["token_type_ids"] = encoding["token_type_ids"][chunk_idx]
            examples.append(ex)
            placed = True
            break

        if not placed:
            skipped += 1

    print(f"Examples created : {len(examples):,}  |  skipped : {skipped}")
    return examples


In [5]:
class QADataset(Dataset):
    """
    Minimal PyTorch Dataset wrapper around the list of tokenised QA examples.
    Each example dict is converted to long tensors on __getitem__.
    """

    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return {k: torch.tensor(v, dtype=torch.long)
                for k, v in self.examples[idx].items()}


## Fine-tune LegalBERT

LegalBERT was pre-trained on 12 GB of legal text (contracts, case law, legislation).
Its domain vocabulary makes it the expected top performer on CUAD.  We add a
randomly-initialised QA head (`BertForQuestionAnswering`) and fine-tune end-to-end.


In [6]:
print(f"Loading LegalBERT tokenizer from: {LEGALBERT_NAME}")
lb_tokenizer = AutoTokenizer.from_pretrained(LEGALBERT_NAME)
print("Done.")


Loading LegalBERT tokenizer from: nlpaueb/legal-bert-base-uncased


Done.


In [7]:
print("Tokenising training sample for LegalBERT...")
lb_examples = make_qa_examples(
    train_sample, lb_tokenizer,
    max_length=MAX_LENGTH,
    stride=STRIDE,
    include_token_type_ids=True,   # BERT-based models use token_type_ids
)
lb_train_ds = QADataset(lb_examples)
print(f"LegalBERT training dataset: {len(lb_train_ds):,} examples")


Tokenising training sample for LegalBERT...


tokenising:   0%|          | 0/2729 [00:00<?, ?it/s]

tokenising:   0%|          | 3/2729 [00:00<01:36, 28.25it/s]

tokenising:   0%|          | 9/2729 [00:00<01:01, 43.95it/s]

tokenising:   1%|          | 15/2729 [00:00<00:55, 48.65it/s]

...

tokenising:  99%|█████████▉| 2707/2729 [00:56<00:00, 91.92it/s]

tokenising: 100%|█████████▉| 2718/2729 [00:57<00:00, 96.20it/s]

tokenising: 100%|██████████| 2729/2729 [00:57<00:00, 99.70it/s]

tokenising: 100%|██████████| 2729/2729 [00:57<00:00, 47.73it/s]

Examples created : 2,610  |  skipped : 119
LegalBERT training dataset: 2,610 examples


In [8]:
lb_model = AutoModelForQuestionAnswering.from_pretrained(LEGALBERT_NAME)
n_params = sum(p.numel() for p in lb_model.parameters())
print(f"LegalBERT parameters: {n_params:,}")


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 78610.78it/s]


[transformers] BertForQuestionAnswering LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	ca

LegalBERT parameters: 108,893,186


In [9]:
lb_training_args = TrainingArguments(
    output_dir=str(LEGALBERT_OUT),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0.1,
    logging_steps=25,
    save_strategy="epoch",
    fp16=False,             # fp16 is not supported on MPS (Apple Silicon)
    report_to="none",       # disable W&B / TensorBoard
    dataloader_num_workers=0,  # avoid macOS fork issues
)
print(f"Training on device: {lb_training_args.device}")


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training on device: mps


In [10]:
lb_trainer = Trainer(
    model=lb_model,
    args=lb_training_args,
    train_dataset=lb_train_ds,
)

print(f"Starting LegalBERT fine-tuning...")
print(f"  Examples : {len(lb_train_ds):,}")
print(f"  Epochs   : {NUM_EPOCHS}")
print()

lb_result = lb_trainer.train()

print("\nLegalBERT training complete.")
print(f"  Final loss    : {lb_result.training_loss:.4f}")
print(f"  Steps trained : {lb_result.global_step:,}")
print(f"  Runtime       : {lb_result.metrics['train_runtime']/60:.1f} min")


Starting LegalBERT fine-tuning...
  Examples : 2,610
  Epochs   : 2



/Users/mustafayunus/Desktop/leaseiq-ml/leaseiq-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
25,5.739376
50,4.681659
75,3.526840
100,2.609702
125,2.810490
150,2.476262
175,2.567273
200,2.560396
225,2.668843
250,2.215487


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.19it/s]

...

/Users/mustafayunus/Desktop/leaseiq-ml/leaseiq-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.35s/it]

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.35s/it]


LegalBERT training complete.
  Final loss    : 2.6795
  Steps trained : 654
  Runtime       : 15.3 min


In [11]:
lb_trainer.save_model(str(LEGALBERT_OUT))
lb_tokenizer.save_pretrained(str(LEGALBERT_OUT))
print(f"LegalBERT model + tokenizer saved to: {LEGALBERT_OUT}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.32it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.31it/s]

LegalBERT model + tokenizer saved to: ../models/legalbert-cuad


## Fine-tune DistilBERT (Baseline)

DistilBERT is a knowledge-distilled version of BERT: **40% fewer parameters** and
**60% faster** inference, retaining ~97% of BERT's GLUE performance.

Key difference from LegalBERT: **DistilBERT does not use `token_type_ids`**.
Setting `include_token_type_ids=False` ensures we do not pass that key to the
model, which would cause a forward-pass error.


In [12]:
print(f"Loading DistilBERT tokenizer from: {DISTILBERT_NAME}")
db_tokenizer = AutoTokenizer.from_pretrained(DISTILBERT_NAME)
print("Done.")


Loading DistilBERT tokenizer from: distilbert-base-uncased


Done.


In [13]:
print("Tokenising training sample for DistilBERT...")
db_examples = make_qa_examples(
    train_sample, db_tokenizer,
    max_length=MAX_LENGTH,
    stride=STRIDE,
    include_token_type_ids=False,  # DistilBERT has no token_type_ids
)
db_train_ds = QADataset(db_examples)
print(f"DistilBERT training dataset: {len(db_train_ds):,} examples")


Tokenising training sample for DistilBERT...


tokenising:   0%|          | 0/2729 [00:00<?, ?it/s]

tokenising:   0%|          | 1/2729 [00:00<44:45,  1.02it/s]

tokenising:   0%|          | 4/2729 [00:01<09:48,  4.63it/s]

tokenising:   0%|          | 8/2729 [00:01<04:40,  9.69it/s]

...

tokenising:  99%|█████████▉| 2702/2729 [01:02<00:00, 92.52it/s]

tokenising:  99%|█████████▉| 2713/2729 [01:02<00:00, 95.73it/s]

tokenising: 100%|█████████▉| 2724/2729 [01:02<00:00, 98.12it/s]

tokenising: 100%|██████████| 2729/2729 [01:02<00:00, 43.70it/s]

Examples created : 2,605  |  skipped : 124
DistilBERT training dataset: 2,605 examples


In [14]:
db_model = AutoModelForQuestionAnswering.from_pretrained(DISTILBERT_NAME)
n_params = sum(p.numel() for p in db_model.parameters())
print(f"DistilBERT parameters: {n_params:,}")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8200.65it/s]


[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT parameters: 66,364,418


In [15]:
db_training_args = TrainingArguments(
    output_dir=str(DISTILBERT_OUT),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0.1,
    logging_steps=25,
    save_strategy="epoch",
    fp16=False,
    report_to="none",
    dataloader_num_workers=0,
)

db_trainer = Trainer(
    model=db_model,
    args=db_training_args,
    train_dataset=db_train_ds,
)

print(f"Starting DistilBERT fine-tuning...")
print(f"  Examples : {len(db_train_ds):,}")
print(f"  Epochs   : {NUM_EPOCHS}")
print()

db_result = db_trainer.train()

print("\nDistilBERT training complete.")
print(f"  Final loss    : {db_result.training_loss:.4f}")
print(f"  Steps trained : {db_result.global_step:,}")
print(f"  Runtime       : {db_result.metrics['train_runtime']/60:.1f} min")


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting DistilBERT fine-tuning...
  Examples : 2,605
  Epochs   : 2



/Users/mustafayunus/Desktop/leaseiq-ml/leaseiq-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
25,5.848817
50,5.091829
75,3.599573
100,3.286393
125,2.950706
150,2.992041
175,2.508578
200,2.789315
225,2.969123
250,2.439261


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/Users/mustafayunus/Desktop/leaseiq-ml/leaseiq-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.37it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.37it/s]


DistilBERT training complete.
  Final loss    : 2.8262
  Steps trained : 652
  Runtime       : 7.3 min


In [16]:
db_trainer.save_model(str(DISTILBERT_OUT))
db_tokenizer.save_pretrained(str(DISTILBERT_OUT))
print(f"DistilBERT model + tokenizer saved to: {DISTILBERT_OUT}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.22s/it]

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.22s/it]

DistilBERT model + tokenizer saved to: ../models/distilbert-cuad


## Training Summary

In [17]:
summary = pd.DataFrame({
    "Model": ["LegalBERT", "DistilBERT (baseline)"],
    "HuggingFace ID": [LEGALBERT_NAME, DISTILBERT_NAME],
    "Parameters": [
        f"{sum(p.numel() for p in lb_model.parameters()):,}",
        f"{sum(p.numel() for p in db_model.parameters()):,}",
    ],
    "Train examples": [len(lb_train_ds), len(db_train_ds)],
    "Epochs": [NUM_EPOCHS, NUM_EPOCHS],
    "Batch size": [BATCH_SIZE, BATCH_SIZE],
    "Final train loss": [
        round(lb_result.training_loss, 4),
        round(db_result.training_loss, 4),
    ],
    "Runtime (min)": [
        round(lb_result.metrics["train_runtime"] / 60, 1),
        round(db_result.metrics["train_runtime"] / 60, 1),
    ],
    "Saved to": [str(LEGALBERT_OUT), str(DISTILBERT_OUT)],
}).set_index("Model")

print("=== Training Summary ===")
print(summary.to_string())


=== Training Summary ===


                                        HuggingFace ID   Parameters  Train examples  Epochs  Batch size  Final train loss  Runtime (min)                   Saved to
Model                                                                                                                                                              
LegalBERT              nlpaueb/legal-bert-base-uncased  108,893,186            2610       2           8            2.6795           15.3   ../models/legalbert-cuad
DistilBERT (baseline)          distilbert-base-uncased   66,364,418            2605       2           8            2.8262            7.3  ../models/distilbert-cuad
